# Phase 2: Metric Refinement and Mentor Feedback

This notebook contains the exploratory scripts written after receiving mentor feedback. It includes the logic for discovering the 225k missing rows (JSON parsing), testing different scatter metrics (relative vs. absolute), evaluating outlier rejection (Z-scores), and testing parameter sensitivity.

## Exploring Metrics

Testing various ways to measure volunteer consistency (Absolute, Relative, Poisson).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json

OUTPUT_DIR = 'Outputs/Metric_Exploration'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading data...")
DATA_PATH = 'sunspot-detectives-classifications.csv'
cols = ['user_name', 'user_ip', 'annotations', 'subject_data']
df = pd.read_csv(DATA_PATH, usecols=['classification_id'] + cols)
df = df.dropna(subset=['annotations', 'subject_data'])[cols]

def extract_count(annotations):
    try:
        val = json.loads(annotations)[0]['value']
        if isinstance(val, (int, float)): return int(val)
        if isinstance(val, str): return int(val.strip()) if val.strip() else None
        if isinstance(val, list) and val:
            iv = val[0].get('value')
            if isinstance(iv, int): return iv
            lb = val[0].get('label')
            if lb is not None: return int(lb)
    except: return None

def extract_filename(subject_data):
    try:
        fv = list(json.loads(subject_data).values())[0]
        parts = fv['Filename'].replace('.png','').split('_')
        return parts[0], parts[1]
    except: return None, None

def assign_volunteer_id(row):
    if pd.notna(row['user_name']): return row['user_name']
    if pd.notna(row['user_ip']):   return 'anon_' + str(row['user_ip'])
    return 'unknown'

print("Parsing classifications...")
df['spot_count'] = df['annotations'].apply(extract_count)
df = df.dropna(subset=['spot_count'])
df['spot_count'] = df['spot_count'].astype(int)
df[['day_id','group_id']] = df['subject_data'].apply(extract_filename).apply(pd.Series)
df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']

# ── CALCULATE METRICS ─────────────────────────────────────────────────────────
print("Calculating metrics against Teolixx...")
teolixx = df[df['volunteer_id'] == 'teolixx'].copy()
others = df[df['volunteer_id'] != 'teolixx'].copy()

teolixx_ref = teolixx.groupby(['day_id','group_id'])['spot_count'].median().reset_index().rename(columns={'spot_count':'teolixx_count'})
merged = pd.merge(others, teolixx_ref, on=['day_id','group_id'])
merged['diff'] = merged['spot_count'] - merged['teolixx_count']

# Calculate group-level std to calculate group-weighted Z-score
group_stats = df.groupby(['day_id','group_id'])['spot_count'].std().reset_index().rename(columns={'spot_count': 'group_std'})
# We only use groups where std > 0 to avoid division by zero
group_stats = group_stats[group_stats['group_std'] > 0]
merged = pd.merge(merged, group_stats, on=['day_id','group_id'], how='inner') # Use inner join to only keep valid groups for this specific metric
merged['group_weighted_zscore'] = (merged['diff'] / merged['group_std']).abs()

stats = merged.groupby('volunteer_id').agg(
    bias=('diff', 'mean'),
    abs_scatter=('diff', 'std'),
    teolixx_mean=('teolixx_count', 'mean'),
    n_overlap=('teolixx_count', 'count'),
    mean_gw_zscore=('group_weighted_zscore', 'mean')
).reset_index()

stats['abs_scatter'] = stats['abs_scatter'].fillna(0)
# 1. Relative Scatter (Current)
stats['relative_scatter'] = stats['abs_scatter'] / stats['teolixx_mean'].replace(0, np.nan)
# 2. Absolute Scatter
# 3. Poisson-Scaled Scatter -> std / sqrt(mean)
stats['poisson_scatter'] = stats['abs_scatter'] / np.sqrt(stats['teolixx_mean'].replace(0, np.nan))

# Filter to reliable vols for plotting
stats = stats[stats['n_overlap'] >= 5]

# ── PLOT METRICS ──────────────────────────────────────────────────────────────
def plot_metric(metric_col, y_label, title, filename, log_y=True, expected_trend="Horizontal band is ideal"):
    plt.figure(figsize=(8, 6))
    plt.scatter(stats['teolixx_mean'], stats[metric_col], alpha=0.4, s=15, color='steelblue')
    plt.xscale('log')
    if log_y:
        plt.yscale('log')
    plt.xlabel('Mean Teolixx Count on Shared Images')
    plt.ylabel(y_label)
    plt.title(f"{title}\n({expected_trend})")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=150)
    plt.close()

plot_metric('relative_scatter', 'Relative Scatter (std / mean)', 
            'Metric 1: Relative Scatter (Current Baseline)', '01_metric_relative_scatter.png',
            expected_trend="Bad: Explodes at low counts, slopes downward.")

plot_metric('abs_scatter', 'Absolute Scatter (std)', 
            'Metric 2: Absolute Scatter', '02_metric_absolute_scatter.png',
            expected_trend="Bad: Explodes at high counts, slopes upward.")

plot_metric('poisson_scatter', 'Poisson-Scaled Scatter (std / sqrt(mean))', 
            'Metric 3: Poisson-Scaled Scatter (Poisson Z-Score)', '03_metric_poisson_scatter.png',
            expected_trend="Good: Flattens the variance across all activity levels.")

plot_metric('mean_gw_zscore', 'Mean Absolute Group-Weighted Z-Score', 
            'Metric 4: Group-Weighted Z-Score', '04_metric_group_weighted_zscore.png', log_y=False,
            expected_trend="Good: Evaluates against Teolixx, weighted by image difficulty.")

print(f"Metrics generated and plotted in: {OUTPUT_DIR}")


## Evaluating Individual Metrics

Specific analysis on why certain volunteers like WRSunset were problematic despite high overlap.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json

OUTPUT_DIR = 'Outputs/Individual_Metric_Evaluation'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading data...")
DATA_PATH = 'sunspot-detectives-classifications.csv'
cols = ['user_name', 'user_ip', 'annotations', 'subject_data']
df = pd.read_csv(DATA_PATH, usecols=['classification_id'] + cols)
df = df.dropna(subset=['annotations', 'subject_data'])[cols]

def extract_count(annotations):
    try:
        val = json.loads(annotations)[0]['value']
        if isinstance(val, (int, float)): return int(val)
        if isinstance(val, str): return int(val.strip()) if val.strip() else None
        if isinstance(val, list) and val:
            iv = val[0].get('value')
            if isinstance(iv, int): return iv
            lb = val[0].get('label')
            if lb is not None: return int(lb)
    except: return None

def extract_filename(subject_data):
    try:
        fv = list(json.loads(subject_data).values())[0]
        parts = fv['Filename'].replace('.png','').split('_')
        return parts[0], parts[1]
    except: return None, None

def assign_volunteer_id(row):
    if pd.notna(row['user_name']): return row['user_name']
    if pd.notna(row['user_ip']):   return 'anon_' + str(row['user_ip'])
    return 'unknown'

print("Parsing classifications...")
df['spot_count'] = df['annotations'].apply(extract_count)
df = df.dropna(subset=['spot_count'])
df['spot_count'] = df['spot_count'].astype(int)
df[['day_id','group_id']] = df['subject_data'].apply(extract_filename).apply(pd.Series)
df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']

print("Isolating Teolixx overlaps...")
teolixx = df[df['volunteer_id'] == 'teolixx'].copy()
others = df[df['volunteer_id'] != 'teolixx'].copy()

teolixx_ref = teolixx.groupby(['day_id','group_id'])['spot_count'].median().reset_index().rename(columns={'spot_count':'teolixx_count'})
merged = pd.merge(others, teolixx_ref, on=['day_id','group_id'])

# Calculate baseline stats to select representative volunteers
stats = merged.groupby('volunteer_id').agg(
    n_overlap=('teolixx_count', 'count'),
    abs_scatter=('spot_count', lambda x: np.std(x - merged.loc[x.index, 'teolixx_count'])),
    teolixx_mean=('teolixx_count', 'mean')
).reset_index()
stats['relative_scatter'] = stats['abs_scatter'] / stats['teolixx_mean'].replace(0, np.nan)
stats = stats.sort_values('n_overlap', ascending=False)

# Select volunteers
vols_to_plot = {}

if 'WRSunset' in stats['volunteer_id'].values:
    vols_to_plot['WRSunset (Overcounter)'] = 'WRSunset'
else:
    # Fallback if WRSunset not present somehow
    vols_to_plot['Highly Active Overcounter'] = stats.iloc[0]['volunteer_id']

good_vols = stats[(stats['n_overlap'] > 50) & (stats['relative_scatter'] < 0.2)]
if not good_vols.empty:
    vols_to_plot['Good Volunteer'] = good_vols.iloc[0]['volunteer_id']

med_vols = stats[(stats['n_overlap'] > 50) & (stats['relative_scatter'] > 0.4) & (stats['volunteer_id'] != vols_to_plot.get('WRSunset (Overcounter)', ''))]
if not med_vols.empty:
    vols_to_plot['Mediocre Volunteer'] = med_vols.iloc[0]['volunteer_id']

print(f"Selected volunteers to plot: {vols_to_plot}")

def huber_loss(diff, delta=2.0):
    abs_diff = np.abs(diff)
    return np.where(abs_diff <= delta, 0.5 * diff**2, delta * abs_diff - 0.5 * delta**2)

def plot_volunteer_metrics(label, vol_id):
    vol_data = merged[merged['volunteer_id'] == vol_id].copy()
    
    x = vol_data['spot_count'].values
    y = vol_data['teolixx_count'].values
    
    # Avoid div by zero
    y_safe_rel = np.where(y == 0, np.nan, y) 
    y_safe_poisson = np.maximum(y, 1)
    
    diff = x - y
    rel_diff = diff / y_safe_rel
    poisson_diff = diff / np.sqrt(y_safe_poisson)
    huber = huber_loss(diff, delta=2.0)
    
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'Pairwise Image Classifications: {label} ({vol_id})\nN={len(x)} shared images with Teolixx', fontsize=16)
    
    # Plot 1: Absolute Difference
    axs[0, 0].scatter(y, diff, alpha=0.5, s=20, color='blue')
    axs[0, 0].axhline(0, color='black', linewidth=1)
    axs[0, 0].set_title('1. Absolute Difference (x - y)')
    axs[0, 0].set_xlabel('Teolixx True Count (y)')
    axs[0, 0].set_ylabel('Absolute Error')
    axs[0, 0].grid(True, alpha=0.3)
    
    # Plot 2: Relative Difference
    axs[0, 1].scatter(y, rel_diff, alpha=0.5, s=20, color='orange')
    axs[0, 1].axhline(0, color='black', linewidth=1)
    axs[0, 1].set_title('2. Relative Difference ((x - y) / y)')
    axs[0, 1].set_xlabel('Teolixx True Count (y)')
    axs[0, 1].set_ylabel('Relative Error')
    axs[0, 1].set_ylim([-1, min(5, np.nanmax(rel_diff) + 0.1)]) # Cap extreme relative errors for readability
    axs[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Poisson-Scaled Difference
    axs[1, 0].scatter(y, poisson_diff, alpha=0.5, s=20, color='green')
    axs[1, 0].axhline(0, color='black', linewidth=1)
    axs[1, 0].set_title('3. Poisson-Scaled Difference ((x - y) / $\sqrt{y}$)')
    axs[1, 0].set_xlabel('Teolixx True Count (y)')
    axs[1, 0].set_ylabel('Poisson Error')
    axs[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Huber Loss
    axs[1, 1].scatter(y, huber, alpha=0.5, s=20, color='red')
    axs[1, 1].set_title('4. Huber Loss ($\delta=2$)')
    axs[1, 1].set_xlabel('Teolixx True Count (y)')
    axs[1, 1].set_ylabel('Loss Value')
    axs[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    safe_label = label.replace(' ', '_').replace('(', '').replace(')', '')
    filename = f"pairwise_metrics_{safe_label}.png"
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=150)
    plt.close()
    print(f"Saved {filename}")

for label, vol_id in vols_to_plot.items():
    plot_volunteer_metrics(label, vol_id)

print(f"All individual pairwise plots saved to: {OUTPUT_DIR}")


## Investigating Anomalies

Debugging scripts to investigate outliers, median errors, and missing rows.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json

OUTPUT_DIR = 'Outputs/Investigate'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load data to investigate 8 (Z-score skew) and 9 (Running Mean)
df_clean = pd.read_csv('Outputs/Final_Run/group_sunspot_numbers.csv')
daily = pd.read_csv('Outputs/Final_Run/daily_sunspot_numbers.csv')

# --- Investigate #9: Running mean above data around index 1800-1900 ---
plt.figure(figsize=(10,4))
start, end = 1750, 1950
subset = daily.iloc[start:end]
plt.plot(subset.index, subset['daily_count'], label='Daily Count', marker='.', alpha=0.5)
smoothed = daily['daily_count'].rolling(window=180, min_periods=1, center=False).mean()
plt.plot(subset.index, smoothed.iloc[start:end], label='180-Session Rolling Mean (Right-aligned)', color='orange')
smoothed_centered = daily['daily_count'].rolling(window=180, min_periods=1, center=True).mean()
plt.plot(subset.index, smoothed_centered.iloc[start:end], label='180-Session Rolling Mean (Centered)', color='red')
plt.title("Investigation #9: Index 1800-1900")
plt.legend()
plt.savefig(os.path.join(OUTPUT_DIR, 'investigate_9.png'))
plt.close()


# --- Investigate #8: Z-score peak at -1 ---
# To do this, we need the raw z-scores. We have to recalculate them quickly since we didn't save them.
DATA_PATH = 'sunspot-detectives-classifications.csv'
cols = ['user_name', 'user_ip', 'annotations', 'subject_data']
df = pd.read_csv(DATA_PATH, usecols=['classification_id'] + cols)
df = df.dropna(subset=['annotations', 'subject_data'])[cols]

def extract_count(annotations):
    try:
        val = json.loads(annotations)[0]['value']
        if isinstance(val, (int, float)): return int(val)
        if isinstance(val, str): return int(val.strip()) if val.strip() else None
        if isinstance(val, list) and val:
            iv = val[0].get('value')
            if isinstance(iv, int): return iv
            lb = val[0].get('label')
            if lb is not None: return int(lb)
    except: return None

def extract_filename(subject_data):
    try:
        fv = list(json.loads(subject_data).values())[0]
        parts = fv['Filename'].replace('.png','').split('_')
        return parts[0], parts[1]
    except: return None, None

def assign_volunteer_id(row):
    if pd.notna(row['user_name']): return row['user_name']
    if pd.notna(row['user_ip']):   return 'anon_' + str(row['user_ip'])
    return 'unknown'

df['spot_count'] = df['annotations'].apply(extract_count)
df = df.dropna(subset=['spot_count'])
df['spot_count'] = df['spot_count'].astype(int)
df[['day_id','group_id']] = df['subject_data'].apply(extract_filename).apply(pd.Series)
df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']

group_stats = df.groupby(['day_id','group_id'])['spot_count'].agg(['mean','std']).reset_index()
group_stats = group_stats[group_stats['std'] > 0]
df_z = df.merge(group_stats, on=['day_id','group_id'])
df_z['z_score'] = (df_z['spot_count'] - df_z['mean']) / df_z['std']

plt.figure(figsize=(10,4))
plt.hist(df_z[df_z['mean'] < 5]['z_score'], bins=50, alpha=0.5, label='Group Mean < 5', density=True)
plt.hist(df_z[df_z['mean'] >= 5]['z_score'], bins=50, alpha=0.5, label='Group Mean >= 5', density=True)
plt.title("Investigation #8: Z-Score distribution split by Group Mean")
plt.legend()
plt.savefig(os.path.join(OUTPUT_DIR, 'investigate_8.png'))
plt.close()

print("Investigation scripts complete.")


In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = 'sunspot-detectives-classifications.csv'
cols = ['user_name', 'user_ip', 'classification_id', 'annotations', 'subject_data']
df = pd.read_csv(DATA_PATH, usecols=cols)

def assign_volunteer_id(row):
    if pd.notna(row['user_name']): return row['user_name']
    if pd.notna(row['user_ip']):   return 'anon_' + str(row['user_ip'])
    return 'unknown'

df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']

print(f"Raw rows (ignoring 'unknown' user): {len(df)}")
activity_raw = df.groupby('volunteer_id').size()
print(f"Median activity (Raw): {activity_raw.median()}")
print(f"Mean activity (Raw): {activity_raw.mean()}")

# Now consider parsed rows
df = df.dropna(subset=['annotations', 'subject_data'])
import json
def extract_count(annotations):
    try:
        val = json.loads(annotations)[0]['value']
        if isinstance(val, (int, float)): return int(val)
        if isinstance(val, str):
            val = val.strip()
            return int(val) if val else None
        if isinstance(val, list) and val:
            iv = val[0].get('value')
            if isinstance(iv, int): return iv
            lb = val[0].get('label')
            if lb is not None: return int(lb)
    except: return None

df['spot_count'] = df['annotations'].apply(extract_count)
df_parsed = df.dropna(subset=['spot_count']).copy()
print(f"Parsed rows: {len(df_parsed)}")
activity_parsed = df_parsed.groupby('volunteer_id').size()
print(f"Median activity (Parsed): {activity_parsed.median()}")
print(f"Mean activity (Parsed): {activity_parsed.mean()}")



## Various Metric Stability Tests

Generating plots to defend the choice of thresholds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
import os

DATA_PATH = 'sunspot-detectives-classifications.csv'
OUTPUT_DIR  = 'Outputs/Final_Run'

# 1. Investigate the "Median is 3" claim
cols = ['user_name', 'user_ip', 'classification_id', 'annotations', 'subject_data']
df = pd.read_csv(DATA_PATH, usecols=cols)

def assign_volunteer_id(row):
    if pd.notna(row['user_name']): return row['user_name']
    if pd.notna(row['user_ip']):   return 'anon_' + str(row['user_ip'])
    return 'unknown'

df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']

activity = df.groupby('volunteer_id').size().sort_values(ascending=False)
median_activity = activity.median()
percent_le_5 = (activity <= 5).mean() * 100
top_50_pct = activity.head(50).sum() / activity.sum() * 100
over_1000 = (activity > 1000).sum()

print(f"--- VOLUNTEER ACTIVITY STATS ---")
print(f"True Median Activity: {median_activity}")
print(f"% with <= 5 images: {percent_le_5:.1f}%")
print(f"Top 50 contribution: {top_50_pct:.1f}%")
print(f"Volunteers > 1000 images: {over_1000}")

# The user's old stats probably came from a subset of the data or before IPs were merged properly.
# The true median is 12. We will output this to a text file for the user.

with open(os.path.join(OUTPUT_DIR, 'defense_stats.txt'), 'w') as f:
    f.write("DEFENSE 1: The 'Median is 3' Discrepancy\n")
    f.write(f"The old stat was likely from an unmerged or early dataset. The TRUE median on the full 767k dataset is {median_activity:.0f}.\n")
    f.write(f"{percent_le_5:.1f}% of volunteers classified 5 or fewer images.\n")
    f.write(f"Top 50 contributed {top_50_pct:.1f}% of the dataset.\n")
    f.write(f"{over_1000} volunteers classified over 1,000 images.\n\n")

# 2. Defending Bias Threshold = 20
# We need the stats dataframe to show how standard error of bias drops with N
stats = pd.read_csv(os.path.join(OUTPUT_DIR, 'volunteer_stats.csv'))
# Standard Error of the Mean (SEM) = std / sqrt(N)
stats['bias_sem'] = stats['scatter'] / np.sqrt(stats['n_overlap'])

# Group by N and get mean SEM
sem_trend = stats.groupby('n_overlap')['bias_sem'].mean()

plt.figure(figsize=(8,5))
plt.scatter(stats['n_overlap'], stats['bias_sem'], alpha=0.3, color='gray', s=10)
# plot moving average
window = 5
smoothed_sem = sem_trend.rolling(window=window, min_periods=1).mean()
plt.plot(smoothed_sem.index, smoothed_sem.values, color='red', linewidth=2, label='Mean Standard Error')
plt.axvline(20, color='black', linestyle='--', label='Threshold N=20')
plt.axhline(1.0, color='blue', linestyle=':', label='Error = 1 Spot')
plt.xlim(0, 100)
plt.ylim(0, 5)
plt.xlabel('Number of Shared Images (N)')
plt.ylabel('Standard Error of Bias Estimate')
plt.title('Why N=20? (Bias Estimate Reliability)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '09_bias_threshold_defense.png'), dpi=150)
plt.close()

# 3. Defending Z-Score Cutoff = 2.5
# We can calculate theoretical fraction of data retained for a normal distribution
z_cutoffs = np.linspace(1, 4, 100)
retained = [norm.cdf(z) - norm.cdf(-z) for z in z_cutoffs]

plt.figure(figsize=(8,5))
plt.plot(z_cutoffs, [r*100 for r in retained], color='steelblue', linewidth=2)
plt.axvline(2.5, color='red', linestyle='--', label=f'Cutoff 2.5 ({ (norm.cdf(2.5)-norm.cdf(-2.5))*100:.2f}%)')
plt.axvline(2.0, color='gray', linestyle=':', label=f'Cutoff 2.0 ({ (norm.cdf(2.0)-norm.cdf(-2.0))*100:.2f}%)')
plt.axvline(3.0, color='black', linestyle=':', label=f'Cutoff 3.0 ({ (norm.cdf(3.0)-norm.cdf(-3.0))*100:.2f}%)')
plt.xlabel('Z-Score Cutoff')
plt.ylabel('% of Data Retained')
plt.title('Why Z=2.5? (Surgically removing only the extreme 1.2%)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, '10_zscore_defense.png'), dpi=150)
plt.close()

print("Defense plots generated.")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import json

OUTPUT_DIR = 'Outputs/Metric_Stability'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Loading data...")
DATA_PATH = 'sunspot-detectives-classifications.csv'
cols = ['user_name', 'user_ip', 'annotations', 'subject_data']
df = pd.read_csv(DATA_PATH, usecols=['classification_id'] + cols)
df = df.dropna(subset=['annotations', 'subject_data'])[cols]

def extract_count(annotations):
    try:
        val = json.loads(annotations)[0]['value']
        if isinstance(val, (int, float)): return int(val)
        if isinstance(val, str): return int(val.strip()) if val.strip() else None
        if isinstance(val, list) and val:
            iv = val[0].get('value')
            if isinstance(iv, int): return iv
            lb = val[0].get('label')
            if lb is not None: return int(lb)
    except: return None

def extract_filename(subject_data):
    try:
        fv = list(json.loads(subject_data).values())[0]
        parts = fv['Filename'].replace('.png','').split('_')
        return parts[0], parts[1]
    except: return None, None

def assign_volunteer_id(row):
    if pd.notna(row['user_name']): return row['user_name']
    if pd.notna(row['user_ip']):   return 'anon_' + str(row['user_ip'])
    return 'unknown'

print("Parsing classifications...")
df['spot_count'] = df['annotations'].apply(extract_count)
df = df.dropna(subset=['spot_count'])
df['spot_count'] = df['spot_count'].astype(int)
df[['day_id','group_id']] = df['subject_data'].apply(extract_filename).apply(pd.Series)
df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']

# Keep track of chronological sessions
unique_days = sorted(df['day_id'].unique())
day_to_idx = {d: i for i, d in enumerate(unique_days)}
df['session_idx'] = df['day_id'].map(day_to_idx)

print("Calculating metrics against Teolixx...")
teolixx = df[df['volunteer_id'] == 'teolixx'].copy()
others = df[df['volunteer_id'] != 'teolixx'].copy()

teolixx_ref = teolixx.groupby(['day_id','group_id'])['spot_count'].median().reset_index().rename(columns={'spot_count':'teolixx_count'})
merged = pd.merge(others, teolixx_ref, on=['day_id','group_id'])
merged['diff'] = merged['spot_count'] - merged['teolixx_count']

# Group-weighted Z-score
group_stats_raw = df.groupby(['day_id','group_id'])['spot_count'].std().reset_index().rename(columns={'spot_count': 'group_std'})
group_stats_raw = group_stats_raw[group_stats_raw['group_std'] > 0]
merged = pd.merge(merged, group_stats_raw, on=['day_id','group_id'], how='inner')
merged['group_weighted_zscore'] = (merged['diff'] / merged['group_std']).abs()


stats = merged.groupby('volunteer_id').agg(
    abs_scatter=('diff', 'std'),
    teolixx_mean=('teolixx_count', 'mean'),
    n_overlap=('teolixx_count', 'count'),
    mean_gw_zscore=('group_weighted_zscore', 'mean')
).reset_index()

stats['abs_scatter'] = stats['abs_scatter'].fillna(0)
stats['relative_scatter'] = stats['abs_scatter'] / stats['teolixx_mean'].replace(0, np.nan)
stats['poisson_scatter'] = stats['abs_scatter'] / np.sqrt(stats['teolixx_mean'].replace(0, np.nan))

# Require minimum overlap of 2 for all tests
stats = stats[stats['n_overlap'] >= 2]

# ── PIPELINE FUNCTION ─────────────────────────────────────────────────────────
def run_pipeline(accepted_volunteers):
    """Runs the pipeline for a set of accepted volunteers and returns the daily smoothed series."""
    acc_vols = set(accepted_volunteers) | {'teolixx'}
    df_work = df[df['volunteer_id'].isin(acc_vols)].copy()
    
    # 1. Group Aggregation
    df_work = df_work.groupby(['volunteer_id','session_idx','day_id','group_id'])['spot_count'].median().reset_index()
    
    # 2. Z-score outlier rejection (within group)
    gs = df_work.groupby(['day_id','group_id'])['spot_count'].agg(['mean','std']).reset_index()
    df_work = df_work.merge(gs, on=['day_id','group_id'])
    df_work['z_score'] = np.where(df_work['std'] > 0, (df_work['spot_count'] - df_work['mean']) / df_work['std'], 0.0)
    df_clean = df_work[df_work['z_score'].abs() <= 2.5]
    
    # 3. Daily aggregation
    gc = df_clean.groupby(['session_idx','day_id','group_id'])['spot_count'].mean().reset_index()
    daily = gc.groupby('session_idx')['spot_count'].sum().reset_index().rename(columns={'spot_count': 'daily_count'})
    
    # Reindex to ensure all sessions are present
    daily = daily.set_index('session_idx').reindex(range(len(unique_days))).fillna(0)
    
    # 4. Smoothing (180-session running mean)
    smoothed = daily['daily_count'].rolling(window=180, min_periods=1, center=True).mean()
    return smoothed

print("Computing baseline...")
baseline_vols = stats[stats['relative_scatter'] <= 0.5]['volunteer_id']
baseline_series = run_pipeline(baseline_vols)

# ── PLOTTING FUNCTION ─────────────────────────────────────────────────────────
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

def plot_stability(metric_col, thresholds, directions, title, filename):
    print(f"Plotting {title}...")
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
    
    ax1.plot(baseline_series.index, baseline_series.values, color='black', linewidth=1.5, alpha=0.3, label='Baseline (RS<=0.5)')
    ax2.axhline(0, color='black', linewidth=1.5, alpha=0.3)
    
    for i, (thresh, dir) in enumerate(zip(thresholds, directions)):
        if dir == '<=':
            acc = stats[stats[metric_col] <= thresh]['volunteer_id']
            label = f'{metric_col} <= {thresh}'
        elif dir == '>=':
            acc = stats[stats[metric_col] >= thresh]['volunteer_id']
            label = f'{metric_col} >= {thresh}'
        
        series = run_pipeline(acc)
        color = colors[i % len(colors)]
        
        ax1.plot(series.index, series.values, color=color, linewidth=1.5, label=label)
        ax2.plot(series.index, series.values - baseline_series.values, color=color, linewidth=1.5, label=label)
    
    ax1.set_ylabel('Smoothed Daily Count (180-Session Mean)')
    ax1.set_title(title)
    ax1.legend()
    
    ax2.set_xlabel('Observing Session (Chronological Index)')
    ax2.set_ylabel('Difference from Baseline')
    ax2.legend()
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=150)
    plt.close()

# 1. Relative Scatter 
plot_stability('relative_scatter', [0.3, 0.5, 0.7], ['<=', '<=', '<='], 
               'Stability: Relative Scatter Criteria', '01_stability_relative_scatter.png')

# 2. Absolute Scatter
plot_stability('abs_scatter', [2.0, 3.0, 5.0], ['<=', '<=', '<='], 
               'Stability: Absolute Scatter Criteria', '02_stability_absolute_scatter.png')

# 3. Poisson Scatter
plot_stability('poisson_scatter', [0.5, 1.0, 1.5], ['<=', '<=', '<='], 
               'Stability: Poisson-Scaled Scatter Criteria', '03_stability_poisson_scatter.png')

# 4. Group-Weighted Z-Score
plot_stability('mean_gw_zscore', [0.5, 1.0, 1.5], ['<=', '<=', '<='], 
               'Stability: Mean Absolute Group-Weighted Z-Score Criteria', '04_stability_group_weighted_zscore.png')

print("All stability plots generated successfully.")


## Scratch & Analysis Drafts

Rough code used during the testing phase.

In [ ]:
import pandas as pd
import json

df = pd.read_csv('sunspot-detectives-classifications.csv', usecols=['user_name', 'user_ip', 'classification_id', 'subject_data'])

def assign_volunteer_id(row):
    if pd.notna(row['user_name']): return row['user_name']
    if pd.notna(row['user_ip']):   return 'anon_' + str(row['user_ip'])
    return 'unknown'

df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)

def extract_filename(subject_data):
    try:
        fv = list(json.loads(subject_data).values())[0]
        parts = fv['Filename'].replace('.png','').split('_')
        return parts[0], parts[1]
    except: return None, None
    
df[['day_id','group_id']] = df['subject_data'].apply(extract_filename).apply(pd.Series)

# Find overlap with teolixx
teolixx_groups = df[df['volunteer_id'] == 'teolixx'][['day_id', 'group_id']].drop_duplicates()
others = df[df['volunteer_id'] != 'teolixx']
merged = pd.merge(others, teolixx_groups, on=['day_id', 'group_id'])
overlap_counts = merged.groupby('volunteer_id').size().sort_values(ascending=False)
print("Top overlapping volunteers:")
print(overlap_counts.head(10))


In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import os
from scipy.stats import norm

# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_PATH   = 'sunspot-detectives-classifications.csv'
OUTPUT_DIR  = 'Outputs/Analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. PARSE DATA
print("Loading and parsing data...")
cols = ['user_name', 'user_ip', 'annotations', 'subject_data']
df = pd.read_csv(DATA_PATH, usecols=['classification_id'] + cols)
df = df.dropna(subset=['annotations', 'subject_data'])[cols]

def extract_count(annotations):
    try:
        val = json.loads(annotations)[0]['value']
        if isinstance(val, (int, float)): return int(val)
        if isinstance(val, str):
            val = val.strip()
            return int(val) if val else None
        if isinstance(val, list) and val:
            iv = val[0].get('value')
            if isinstance(iv, int): return iv
            lb = val[0].get('label')
            if lb is not None: return int(lb)
    except: return None

def extract_filename(subject_data):
    try:
        fv = list(json.loads(subject_data).values())[0]
        parts = fv['Filename'].replace('.png','').split('_')
        return parts[0], parts[1]
    except: return None, None

def assign_volunteer_id(row):
    if pd.notna(row['user_name']): return row['user_name']
    if pd.notna(row['user_ip']):   return 'anon_' + str(row['user_ip'])
    return 'unknown'

df['spot_count']   = df['annotations'].apply(extract_count)
df = df.dropna(subset=['spot_count'])
df['spot_count']   = df['spot_count'].astype(int)
df[['day_id','group_id']] = df['subject_data'].apply(extract_filename).apply(pd.Series)
df['volunteer_id'] = df.apply(assign_volunteer_id, axis=1)
df = df[df['volunteer_id'] != 'unknown']
df = df.drop(columns=['annotations','subject_data','user_name','user_ip'])

# 2. TEOLIXX REFERENCE
teolixx = df[df['volunteer_id'] == 'teolixx'].copy()
others  = df[df['volunteer_id'] != 'teolixx'].copy()

teolixx_ref = (teolixx.groupby(['day_id','group_id'])['spot_count']
               .median().reset_index()
               .rename(columns={'spot_count':'teolixx_count'}))

merged = pd.merge(others, teolixx_ref, on=['day_id','group_id'])
merged['diff'] = merged['spot_count'] - merged['teolixx_count']

stats = (merged.groupby('volunteer_id')['diff']
         .agg(['mean','std','count']).reset_index()
         .rename(columns={'mean':'bias','std':'scatter','count':'n_overlap'}))
stats['scatter'] = stats['scatter'].fillna(0)

teolixx_mean = (merged.groupby('volunteer_id')['teolixx_count']
                .mean().reset_index()
                .rename(columns={'teolixx_count':'teolixx_mean_count'}))
stats = stats.merge(teolixx_mean, on='volunteer_id')
stats['relative_scatter'] = stats['scatter'] / stats['teolixx_mean_count']

# --- Task 3: Investigate volunteer with ~2000 overlapping images ---
print("\n--- Investigating high-overlap volunteers ---")
print(stats.sort_values('n_overlap', ascending=False).head(10)[['volunteer_id', 'n_overlap', 'bias', 'scatter', 'relative_scatter']])

strange_vol = stats.sort_values('n_overlap', ascending=False).iloc[0]
print(f"\nMost overlapping volunteer: {strange_vol['volunteer_id']}")
# Plot timeseries for this volunteer vs teolixx
vol_data = merged[merged['volunteer_id'] == strange_vol['volunteer_id']].copy()
# Aggregate to daily
vol_daily = vol_data.groupby('day_id').agg({'spot_count': 'sum', 'teolixx_count': 'sum'}).reset_index()
vol_daily = vol_daily.sort_values('day_id')

plt.figure(figsize=(12, 4))
plt.plot(range(len(vol_daily)), vol_daily['teolixx_count'], label='Teolixx', alpha=0.7)
plt.plot(range(len(vol_daily)), vol_daily['spot_count'], label=strange_vol['volunteer_id'], alpha=0.7)
plt.xlabel('Day Index')
plt.ylabel('Daily Sunspot Count')
plt.title(f'Counts comparison: {strange_vol["volunteer_id"]} vs Teolixx')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'strange_volunteer_timeseries.png'))
plt.close()

# --- Task 4: Metrics at low vs high counts ---
# Plot relative_scatter vs teolixx_mean_count
plt.figure(figsize=(8, 6))
plt.scatter(stats['teolixx_mean_count'], stats['relative_scatter'], alpha=0.5, s=10)
plt.xlabel('Mean Teolixx Count (Shared Images)')
plt.ylabel('Relative Scatter')
plt.title('Relative Scatter vs. Count Magnitude')
plt.yscale('log')
plt.xscale('log')
plt.axhline(0.5, color='r', linestyle='--', label='Threshold (0.5)')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'metrics_low_vs_high_counts.png'))
plt.close()

# --- Task 5 & 7: Stability Plot and Curve Comparisons ---
# We will run the pipeline for different parameter sets
def run_pipeline(min_overlap, max_rel_scatter):
    acc_mask = (stats['n_overlap'] >= min_overlap) & (stats['relative_scatter'] <= max_rel_scatter)
    acc_stats = stats[acc_mask].copy()
    acc_ids = acc_stats['volunteer_id'].tolist() + ['teolixx']
    
    df_w = df[df['volunteer_id'].isin(acc_ids)].copy()
    
    # Simple pipeline without bias correction or outlier removal for rapid comparison
    group_counts = (df_w.groupby(['day_id','group_id'])['spot_count']
                    .agg(['mean']).reset_index()
                    .rename(columns={'mean':'mean_count'}))
    
    daily = group_counts.groupby('day_id')['mean_count'].sum().reset_index()
    return daily.set_index('day_id')['mean_count']

# Get common timeline
all_days = sorted(df['day_id'].unique())

strict = run_pipeline(5, 0.3)
current = run_pipeline(5, 0.5)
loose = run_pipeline(5, 0.7)
looser_overlap = run_pipeline(2, 0.5) # Task 6: Looser overlap (2 images)

comb = pd.DataFrame(index=all_days)
comb['strict'] = strict
comb['current'] = current
comb['loose'] = loose
comb['looser_overlap'] = looser_overlap
comb = comb.fillna(0)

print("\n--- Curve Comparisons (RMSE and Correlation vs Current) ---")
for col in ['strict', 'loose', 'looser_overlap']:
    rmse = np.sqrt(np.mean((comb['current'] - comb[col])**2))
    corr = comb['current'].corr(comb[col])
    print(f"{col:>15}: RMSE={rmse:.2f}, Correlation={corr:.4f}")

# Plot stability with aligned x-axis and smoothed panel
window = 180
smoothed = comb.rolling(window=window, min_periods=1).mean()

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(15, 12), sharex=True)

# Raw
x_idx = np.arange(len(comb))
ax1.plot(x_idx, comb['strict'], label='Strict (n>=5, rs<=0.3)', alpha=0.5, linewidth=0.8)
ax1.plot(x_idx, comb['current'], label='Current (n>=5, rs<=0.5)', alpha=0.5, linewidth=0.8)
ax1.plot(x_idx, comb['loose'], label='Loose (n>=5, rs<=0.7)', alpha=0.5, linewidth=0.8)
ax1.set_ylabel('Daily Sunspot Count')
ax1.set_title('Daily Count Stability (Raw)')
ax1.legend()

# Smoothed
ax2.plot(x_idx, smoothed['strict'], label='Strict (n>=5, rs<=0.3)')
ax2.plot(x_idx, smoothed['current'], label='Current (n>=5, rs<=0.5)')
ax2.plot(x_idx, smoothed['loose'], label='Loose (n>=5, rs<=0.7)')
ax2.plot(x_idx, smoothed['looser_overlap'], label='Looser Overlap (n>=2, rs<=0.5)', linestyle='--')
ax2.set_ylabel(f'Smoothed ({window}-day MA)')
ax2.set_title(f'Daily Count Stability ({window}-day Running Mean)')
ax2.legend()

# Difference from current
ax3.plot(x_idx, (smoothed['strict'] - smoothed['current']), label='Strict - Current')
ax3.plot(x_idx, (smoothed['loose'] - smoothed['current']), label='Loose - Current')
ax3.plot(x_idx, (smoothed['looser_overlap'] - smoothed['current']), label='Looser Overlap - Current', linestyle='--')
ax3.set_ylabel('Difference (Smoothed)')
ax3.set_xlabel('Observing Session (Chronological Index)')
ax3.set_title('Difference from Current Parameters (Smoothed)')
ax3.axhline(0, color='k', linestyle='--', linewidth=0.8)
ax3.legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'stability_analysis.png'))
plt.close()

print("\nSaved all plots to Outputs/Analysis/")
